# 12 pi0 strict-input 端到端诊断

        这一节把 pi0 调试中最容易混淆的四件事拆开：raw policy、只用 VLA 合法输入的 learned head、固定场景策略采样稳定性、真正随机物体位置泛化。所有表格都来自 AMD 设备实测结果。


In [1]:

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    override = (
        os.environ.get("AMD_TOPIC_ROOT")
        or os.environ.get("NOTEBOOK_TOPIC_ROOT")
        or os.environ.get("TOPIC_ROOT")
    )
    roots = [Path(override).expanduser()] if override else []
    cwd = Path.cwd().resolve()
    roots.extend([cwd, *cwd.parents])

    candidates = []
    for root in roots:
        candidates.extend(
            [
                root,
                root / "16-专题组队学习" / "04-AMD-ROCm策略复刻专题",
                root / "04-AMD-ROCm策略复刻专题",
            ]
        )
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError(
        "找不到 AMD ROCm 专题目录。请从仓库根目录、专题目录启动 Jupyter，"
        "或设置 AMD_TOPIC_ROOT。"
    )


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(
    os.environ.get("PROJECT_ROOT", TOPIC_ROOT / "external" / "mujoco_pnp")
).expanduser()
DATA_ROOT = Path(os.environ.get("DATA_ROOT", TOPIC_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


TOPIC_ROOT = $NOTEBOOK_TOPIC_ROOT
PROJECT_ROOT = $PROJECT_ROOT
DATA_ROOT = $PROJECT_ROOT
OUTPUT_ROOT = $OUTPUT_ROOT
MODEL_ROOT = $MODEL_ROOT


In [2]:

try:
    from IPython.display import Image, Markdown, Video, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"

    def Video(filename=None, embed=False, width=None, **kwargs):
        return f"[video] {filename}"


def show_video(filename, title=None, width=960):
    path = ASSET_DIR / filename
    if title:
        display(Markdown(f"**{title}**"))
    if not path.exists():
        print(f"缺少视频素材：{path}")
        return
    try:
        display(Video(filename=str(path), embed=True, width=width, html_attributes="controls muted"))
    except TypeError:
        display(Video(filename=str(path), embed=True, width=width))


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        if path.suffix.lower() in {".mp4", ".webm", ".mov", ".m4v"}:
            show_video(filename, width=width)
            return
        display(Image(filename=str(path), width=width))
    else:
        print(f"缺少素材：{path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1：读取最终严格评估结果


In [3]:
results = json.loads(
    (ASSET_DIR / "pi0_strict_input_results.json").read_text(encoding="utf-8")
)
fixed = results["fixed_scene"]
randomized = results["randomized_environment_gate"]
rows = [
    (
        "raw pi0",
        "env seed 0；12 个 policy sampling seeds",
        f'{fixed["raw_pi0"]["success"]}/{fixed["raw_pi0"]["total"]}',
    ),
    (
        "pi0 + visual/history GRU head",
        "env seed 0；同一批 12 个 policy sampling seeds",
        f'{fixed["pi0_visual_history_gru"]["success"]}/{fixed["pi0_visual_history_gru"]["total"]}',
    ),
    (
        "pi0 + visual/history GRU head",
        "修正 seed 后的随机环境 41–44",
        f'{randomized["pi0_visual_history_gru"]["success"]}/{randomized["pi0_visual_history_gru"]["total"]}',
    ),
]
md_table(["策略", "协议", "final strict"], rows)


| 策略 | 协议 | final strict |
| --- | --- | --- |
| raw pi0 | env seed 0；12 个 policy sampling seeds | 0/12 |
| pi0 + visual/history GRU head | env seed 0；同一批 12 个 policy sampling seeds | 6/12 |
| pi0 + visual/history GRU head | 修正 seed 后的随机环境 41–44 | 1/4 |


![pi0 strict-input 对照](../assets/pi0_strict_input_progress.png)

        固定场景从 `0/12` 提到 `6/12` 是实质进步，但不能跳过随机环境 `1/4`。这也是为什么教程把“固定场景拟合”和“位置泛化”分开写。


## Checkpoint 2：learned head 到底看了什么


In [4]:
rows = [
    ("允许", "agent/wrist 图像、语言、6D robot proprio、上一条策略自己执行过的 7D EEF/gripper 命令"),
    ("禁止", "target xyz、plate xyz、GT phase、oracle action、当前或未来 GT gripper 标签"),
    ("输出", "EEF keyframe/历史动作与 gripper close probability"),
    ("部署", "gripper probability threshold=0.5，再转成二值执行命令"),
]
md_table(["类别", "内容"], rows)


| 类别 | 内容 |
| --- | --- |
| 允许 | agent/wrist 图像、语言、6D robot proprio、上一条策略自己执行过的 7D EEF/gripper 命令 |
| 禁止 | target xyz、plate xyz、GT phase、oracle action、当前或未来 GT gripper 标签 |
| 输出 | EEF keyframe/历史动作与 gripper close probability |
| 部署 | gripper probability threshold=0.5，再转成二值执行命令 |


这个 head 具备 VLA-compatible 的输入边界，但它是额外训练的视觉/历史控制头，因此报告时写 `pi0 + learned head`，不能写成 raw pi0。它学的是图像与机器人历史中的阶段线索，不依赖鲜艳标记或手工目标坐标。


## Checkpoint 3：visual/history head 的实际训练命令


In [5]:
import shlex

HEAD_SOURCE_REPO = os.environ.get("HEAD_SOURCE_REPO", "your_multiseed_fulltask_dataset")
HEAD_SOURCE_ROOT = Path(os.environ.get("HEAD_SOURCE_ROOT", DATA_ROOT / HEAD_SOURCE_REPO))
HEAD_SOURCE_JSONL = Path(os.environ.get("HEAD_SOURCE_JSONL", OUTPUT_ROOT / "collection_results.jsonl"))
HEAD_POLICY_PATH = Path(os.environ.get("HEAD_POLICY_PATH", MODEL_ROOT / "pi0_base" / "pretrained_model"))
HEAD_DIR = Path(os.environ.get("HEAD_DIR", MODEL_ROOT / "pi0_visual_history_head"))
FEATURE_CACHE = HEAD_DIR / "visual_features.npz"
DIRECT_HEAD = HEAD_DIR / "visual_keyframe_head.npz"
GRU_HEAD = HEAD_DIR / "visual_keyframe_gru_head.npz"
RUN_AUXILIARY_TRAIN = False

commands = [
    [
        sys.executable,
        str(TOPIC_ROOT / "code" / "pi0" / "train_pi0_visual_contact_head.py"),
        "--source", HEAD_SOURCE_REPO, str(HEAD_SOURCE_ROOT), str(HEAD_SOURCE_JSONL),
        "--policy-path", str(HEAD_POLICY_PATH),
        "--stats-repo-id", HEAD_SOURCE_REPO,
        "--stats-dataset-root", str(HEAD_SOURCE_ROOT),
        "--train-seeds", "21-32",
        "--val-seeds", "33-36",
        "--append-prev-action",
        "--feature-cache", str(FEATURE_CACHE),
        "--output", str(DIRECT_HEAD),
        "--summary-json", str(HEAD_DIR / "visual_keyframe_head_summary.json"),
    ],
    [
        sys.executable,
        str(TOPIC_ROOT / "code" / "pi0" / "train_pi0_visual_gripper_gru.py"),
        "--source", HEAD_SOURCE_REPO, str(HEAD_SOURCE_ROOT), str(HEAD_SOURCE_JSONL),
        "--feature-cache", str(FEATURE_CACHE),
        "--direct-head", str(DIRECT_HEAD),
        "--train-seeds", "21-32",
        "--val-seeds", "33-36",
        "--epochs", "300",
        "--transition-weight", "20",
        "--release-weight", "40",
        "--output", str(GRU_HEAD),
        "--summary-json", str(HEAD_DIR / "visual_keyframe_gru_summary.json"),
    ],
    [
        sys.executable,
        str(TOPIC_ROOT / "code" / "pi0" / "test_pi0_visual_gripper_gru_parity.py"),
        "--head", str(GRU_HEAD),
        "--steps", "32",
    ],
]

for command in commands:
    print("$", shlex.join(command))
if RUN_AUXILIARY_TRAIN:
    HEAD_DIR.mkdir(parents=True, exist_ok=True)
    for command in commands:
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("默认只预览。先替换多位置数据、collector JSONL 和 base policy 路径，再打开 RUN_AUXILIARY_TRAIN。")


$ $NOTEBOOK_PYTHON $NOTEBOOK_TOPIC_ROOT/code/pi0/train_pi0_visual_contact_head.py --source your_multiseed_fulltask_dataset '$PROJECT_ROOT/your_multiseed_fulltask_dataset' '$OUTPUT_ROOT/collection_results.jsonl' --policy-path '$MODEL_ROOT/pi0_base/pretrained_model' --stats-repo-id your_multiseed_fulltask_dataset --stats-dataset-root '$PROJECT_ROOT/your_multiseed_fulltask_dataset' --train-seeds 21-32 --val-seeds 33-36 --append-prev-action --feature-cache '$MODEL_ROOT/pi0_visual_history_head/visual_features.npz' --output '$MODEL_ROOT/pi0_visual_history_head/visual_keyframe_head.npz' --summary-json '$MODEL_ROOT/pi0_visual_history_head/visual_keyframe_head_summary.json'
$ $NOTEBOOK_PYTHON $NOTEBOOK_TOPIC_ROOT/code/pi0/train_pi0_visual_gripper_gru.py --source your_multiseed_fulltask_dataset '$PROJECT_ROOT/your_multiseed_fulltask_dataset' '$OUTPUT_ROOT/collection_results.jsonl' --feature-cache '$MODEL_ROOT/pi0_visual_history_head/visual_features.npz' --direct-head '$MODEL_ROOT/pi0_visual_hist

## Checkpoint 4：语言表述也属于部署协议


In [6]:
rows = [
    (
        "Place the blue mug on the plate.",
        "0 active steps",
        "0/1",
        "短 prompt 超出小头训练分布，prototype gate 一直拒绝接管",
    ),
    (
        "Pick up the blue mug and place it on the plate.",
        "269 active steps",
        "1/1",
        "与训练指令一致；seed3 在 270 步通过 final strict",
    ),
]
md_table(["instruction", "learned head", "strict", "现象"], rows)


| instruction | learned head | strict | 现象 |
| --- | --- | --- | --- |
| Place the blue mug on the plate. | 0 active steps | 0/1 | 短 prompt 超出小头训练分布，prototype gate 一直拒绝接管 |
| Pick up the blue mug and place it on the plate. | 269 active steps | 1/1 | 与训练指令一致；seed3 在 270 步通过 final strict |


这不是说 VLA 永远不能接受同义指令，而是当前 4 条小数据训练出来的 auxiliary head 还没有语言改写泛化。部署时要先固定训练/评估 prompt；下一轮再加入同义指令增强，并单独测试 language paraphrase gate。


## Checkpoint 5：为什么 overall accuracy 会骗人


In [7]:
rows = [
    ("旧线性 gripper head", "99.66%", "0/4", "release 只有极少帧，总体准确率掩盖了全漏"),
    ("GRU sequence head", "96.8% balanced", "5/5", "按 close/release transition 单独选阈值"),
]
md_table(["模型", "验证指标", "held-out release", "解读"], rows)


| 模型 | 验证指标 | held-out release | 解读 |
| --- | --- | --- | --- |
| 旧线性 gripper head | 99.66% | 0/4 | release 只有极少帧，总体准确率掩盖了全漏 |
| GRU sequence head | 96.8% balanced | 5/5 | 按 close/release transition 单独选阈值 |


## Checkpoint 6：用代码复现最终成功谓词


In [8]:
def final_strict_success(row):
    p = results["strict_predicate"]
    return bool(
        row["legacy_success"]
        and row["max_target_lift"] >= p["min_lift_m"]
        and row["max_lifted_run"] >= p["min_lift_steps"]
        and row["upright_cos"] >= p["min_upright_cos"]
        and row["plate_z_gap"] <= p["max_plate_z_gap_m"]
        and row["plate_xy_displacement"] <= p["max_plate_xy_displacement_m"]
        and row["stable_place_steps"] >= p["stable_place_steps"]
    )


known_success = {
    "legacy_success": True,
    "max_target_lift": 0.11,
    "max_lifted_run": 8,
    "upright_cos": 0.95,
    "plate_z_gap": 0.0448,
    "plate_xy_displacement": 0.0105,
    "stable_place_steps": 5,
}
print("示例是否通过最终严格判定：", final_strict_success(known_success))


示例是否通过最终严格判定： True


## Checkpoint 7：seed bug 对结论的影响


In [9]:
seed_bug = results["seed_bug"]
print("旧代码：", seed_bug["old_code"])
print("修复后：", seed_bug["fixed_code"])
print("影响：", seed_bug["impact"])


旧代码： if seed != None: np.random.seed(seed=0)
修复后： if seed is not None: np.random.seed(seed=seed)
影响： 旧 30-seed 面板只改变策略采样随机性，环境始终是 seed 0，不能作为空间泛化证据


## Checkpoint 8：四视角成功回放

        ![四视角成功回放关键帧](../assets/pnp_four_view_strict_success_sequence.jpg)

        <video controls width="960" src="../assets/pnp_four_view_strict_success.mp4"></video>


## Checkpoint 9：下一轮数据计划


In [10]:
plan = [
    ("采集", "修正 seed 后采 30–50 条多位置完整成功轨迹"),
    ("分层", "按红/蓝杯、起点区域、contact miss、transport tip、release fail 分桶"),
    ("训练", "先保护 fixed-scene 6/12，再训练 visual/history head；不要继续盲加 raw pi0 steps"),
    ("评估", "fixed scene 与 random env 分开；随机环境至少 20–30 条"),
    ("替换门槛", "新模型同协议超过保护基线，且视频无推杯/悬空误判"),
]
md_table(["阶段", "动作"], plan)


| 阶段 | 动作 |
| --- | --- |
| 采集 | 修正 seed 后采 30–50 条多位置完整成功轨迹 |
| 分层 | 按红/蓝杯、起点区域、contact miss、transport tip、release fail 分桶 |
| 训练 | 先保护 fixed-scene 6/12，再训练 visual/history head；不要继续盲加 raw pi0 steps |
| 评估 | fixed scene 与 random env 分开；随机环境至少 20–30 条 |
| 替换门槛 | 新模型同协议超过保护基线，且视频无推杯/悬空误判 |
